# Plan-And-Solve

ReAct是一种边思考边行动的泛式，与之不同的是，Plan-And-Solve先进行思考，形成一个完整的计划，然后再执行这个计划。Plan-And-Solve的核心思想是将问题分解为两个阶段：计划阶段和执行阶段。在计划阶段，模型会生成一个详细的计划，包括每一步的具体操作和预期结果。在执行阶段，模型会按照计划进行操作，并根据实际情况进行调整。

## Plan-And-Solve原理
Plan-And-Solve的原理可以概括为以下几个步骤：
1. **问题分析**：模型首先对输入的问题进行分析，理解问题的背景和要求。
2. **计划生成**：模型根据问题分析的结果，生成一个详细的计划。这个计划包括每一步的具体操作和预期结果。
3. **计划执行**：模型按照生成的计划进行操作。它会严格按照计划执行，每一次执行都是一次独立的LLM调用。

这种先谋后动的方式能让智能体更加长远地思考问题，能保持高度的目标一致性。

Plan-And-Solve适合结构性强，可以被清晰分解的复杂任务，例如：
- **多步的数学问题**：需要多个步骤才能得出最终答案的问题。
- **整合资源的报告撰写**： 需要整合多个信息来源来撰写报告的任务。
- **代码生成任务**：需要先构思好函数类模块等结构，再进行代码编写的任务。

## Plan-And-Solve示例
我们来设定一个目标问题：“一个水果店周一卖出15个苹果，周二卖出是苹果数量是周一的两倍。周三卖出数量比周二少5个。请问三天一共卖出了多少个苹果？”

### 计划阶段
1. **分析问题**：我们需要计算三天卖出的苹果总数。首先，我们知道周一卖出了15个苹果。周二卖出的数量是周一的两倍，所以周二卖出了30个苹果。周三卖出的数量比周二少5个，所以周三卖出了25个苹果。
2. **生成计划**：我们可以将问题分解为以下步骤：
   - 计算周二卖出的苹果数量：周二 = 周一 * 2
   - 计算周三卖出的苹果数量：周三 = 周二 - 5
   - 计算三天卖出的总数：总数 = 周一 + 周二 + 周三

规划阶段我们接收问题，分析问题，生成计划，输出计划。因此我们需要设计一个Prompt来引导模型进行计划生成。

In [1]:
PLANNER_PROMPT_TEMPLATE = """
你是一个顶级的AI规划专家。你的任务是将用户提出的复杂问题分解成一个由多个简单步骤组成的行动计划。
请确保计划中的每个步骤都是一个独立的、可执行的子任务，并且严格按照逻辑顺序排列。
你的输出必须是一个Python列表，其中每个元素都是一个描述子任务的字符串。

问题: {question}

请严格按照以下格式输出你的计划,```python与```作为前后缀是必要的:
```python
["步骤1", "步骤2", "步骤3", ...]
```
"""


这个提示词通过一下几点确保了输出的质量和稳定性：
- **角色设定**：明确告诉模型它是一个顶级的AI规划专家，这有助于提升输出的专业性和准确性。
- **任务描述**：清晰地描述了模型的任务，即将复杂问题分解成一个由多个简单步骤组成的行动计划。
- **输出格式**：明确要求模型输出一个Python列表，并且每个元素都是一个描述子任务的字符串。这种格式化的要求有助于确保输出的一致性和可解析性。

接下来我们来定义一个Planner类来使用这个提示词进行计划生成。

In [2]:
import ast
# 假定 llm_client.py 中的 HelloAgentsLLM 类已经定义好
from utils.llm import HelloAgentsLLM

class Planner:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def plan(self, question: str) -> list[str]:
        """
        根据用户问题生成一个行动计划。
        """
        prompt = PLANNER_PROMPT_TEMPLATE.format(question=question)

        # 为了生成计划，我们构建一个简单的消息列表
        messages = [{"role": "user", "content": prompt}]

        print("--- 正在生成计划 ---")
        # 使用流式输出来获取完整的计划
        response_text = self.llm_client.think(messages=messages) or ""

        print(f"✅ 计划已生成:\n{response_text}")

        # 解析LLM输出的列表字符串
        try:
            # 找到```python和```之间的内容
            plan_str = response_text.split("```python")[1].split("```")[0].strip()
            # 使用ast.literal_eval来安全地执行字符串，将其转换为Python列表
            plan = ast.literal_eval(plan_str)
            return plan if isinstance(plan, list) else []
        except (ValueError, SyntaxError, IndexError) as e:
            print(f"❌ 解析计划时出错: {e}")
            print(f"原始响应: {response_text}")
            return []
        except Exception as e:
            print(f"❌ 解析计划时发生未知错误: {e}")
            return []


## 执行器和状态管理

在规划器生成清晰的行动蓝图后，我们需要一个执行器来逐一执行计划任务，还需要承担一个重要角色：状态管理器。状态管理器负责跟踪和更新执行过程中产生的状态信息，确保每一步操作都能基于最新的上下文进行。

提示词需要包含以下关键信息：
- **原始问题**：明确告诉模型它需要解决的问题是什么。
- **完整计划**：提供规划器生成的完整计划，让模型知道接下来要执行的任务和任务所在位置。
- **历史步骤和结果**：提供已经执行过的步骤和对应的结果，作为当前步骤的输入。
- **当前步骤**：明确告诉模型当前需要执行的步骤是什么，以及它在整个计划中的位置。
-

In [3]:
EXECUTOR_PROMPT_TEMPLATE = """
你是一位顶级的AI执行专家。你的任务是严格按照给定的计划，一步步地解决问题。
你将收到原始问题、完整的计划、以及到目前为止已经完成的步骤和结果。
请你专注于解决“当前步骤”，并仅输出该步骤的最终答案，不要输出任何额外的解释或对话。

# 原始问题:
{question}

# 完整计划:
{plan}

# 历史步骤与结果:
{history}

# 当前步骤:
{current_step}

请仅输出针对“当前步骤”的回答:
"""


我们将定义一个Executor类来使用这个提示词进行计划执行。

In [4]:
class Executor:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def execute(self, question: str, plan: list[str]) -> str:
        """
        根据计划，逐步执行并解决问题。
        """
        history = "" # 用于存储历史步骤和结果的字符串

        print("\n--- 正在执行计划 ---")

        for i, step in enumerate(plan):
            print(f"\n-> 正在执行步骤 {i+1}/{len(plan)}: {step}")

            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                question=question,
                plan=plan,
                history=history if history else "无", # 如果是第一步，则历史为空
                current_step=step
            )

            messages = [{"role": "user", "content": prompt}]

            response_text = self.llm_client.think(messages=messages) or ""

            # 更新历史记录，为下一步做准备
            history += f"步骤 {i+1}: {step}\n结果: {response_text}\n\n"

            print(f"✅ 步骤 {i+1} 已完成，结果: {response_text}")

        # 循环结束后，最后一步的响应就是最终答案
        final_answer = response_text
        return final_answer


现在已经分别构建了负责“规划”的 Planner 和负责“执行”的 Executor。最后一步是将这两个组件整合到一个统一的智能体 PlanAndSolveAgent 中，并赋予它解决问题的完整能力。我们将创建一个主类 PlanAndSolveAgent，它的职责非常清晰：接收一个 LLM 客户端，初始化内部的规划器和执行器，并提供一个简单的 run 方法来启动整个流程。

In [5]:
class PlanAndSolveAgent:
    def __init__(self, llm_client):
        """
        初始化智能体，同时创建规划器和执行器实例。
        """
        self.llm_client = llm_client
        self.planner = Planner(self.llm_client)
        self.executor = Executor(self.llm_client)

    def run(self, question: str):
        """
        运行智能体的完整流程:先规划，后执行。
        """
        print(f"\n--- 开始处理问题 ---\n问题: {question}")

        # 1. 调用规划器生成计划
        plan = self.planner.plan(question)

        # 检查计划是否成功生成
        if not plan:
            print("\n--- 任务终止 --- \n无法生成有效的行动计划。")
            return

        # 2. 调用执行器执行计划
        final_answer = self.executor.execute(question, plan)

        print(f"\n--- 任务完成 ---\n最终答案: {final_answer}")


这个 PlanAndSolveAgent 类的设计体现了“组合优于继承”的原则。它本身不包含复杂的逻辑，而是作为一个协调者 (Orchestrator)，清晰地调用其内部组件来完成任务。

In [6]:
agent = PlanAndSolveAgent(llm_client=HelloAgentsLLM())
question = "一个水果店周一卖出15个苹果，周二卖出是苹果数量是周一的两倍。周三卖出数量比周二少5个。请问三天一共卖出了多少个苹果？"
agent.run(question)


--- 开始处理问题 ---
问题: 一个水果店周一卖出15个苹果，周二卖出是苹果数量是周一的两倍。周三卖出数量比周二少5个。请问三天一共卖出了多少个苹果？
--- 正在生成计划 ---
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
```python
["计算周一卖出的苹果数量：15个", "计算周二卖出的苹果数量：周一数量的两倍，即15 * 2 = 30个", "计算周三卖出的苹果数量：比周二少5个，即30 - 5 = 25个", "计算三天卖出的苹果总数：15 + 30 + 25 = 70个"]
```
✅ 计划已生成:
```python
["计算周一卖出的苹果数量：15个", "计算周二卖出的苹果数量：周一数量的两倍，即15 * 2 = 30个", "计算周三卖出的苹果数量：比周二少5个，即30 - 5 = 25个", "计算三天卖出的苹果总数：15 + 30 + 25 = 70个"]
```

--- 正在执行计划 ---

-> 正在执行步骤 1/4: 计算周一卖出的苹果数量：15个
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
周一卖出的苹果数量是15个。
✅ 步骤 1 已完成，结果: 周一卖出的苹果数量是15个。

-> 正在执行步骤 2/4: 计算周二卖出的苹果数量：周一数量的两倍，即15 * 2 = 30个
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
周二卖出的苹果数量是30个。
✅ 步骤 2 已完成，结果: 周二卖出的苹果数量是30个。

-> 正在执行步骤 3/4: 计算周三卖出的苹果数量：比周二少5个，即30 - 5 = 25个
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
周三卖出的苹果数量是25个。
✅ 步骤 3 已完成，结果: 周三卖出的苹果数量是25个。

-> 正在执行步骤 4/4: 计算三天卖出的苹果总数：15 + 30 + 25 = 70个
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
70个
✅ 步骤 4 已完成，结果: 70个

--- 任务完成 ---
最终答案: 70个
